## Environment and Model setup and configuration

#### Importing data

In [2]:
import os

os.listdir("/kaggle/input/")

['tokenized-bert-reviews-256',
 'bert-sentiment-frozen0',
 '256-tokenizer',
 '512-tokenizer',
 '512-tokenized-reviews']

In [3]:
from datasets import load_from_disk
tokenized = load_from_disk("/kaggle/input/512-tokenized-reviews/tokenized_bert_reviews_512")

#### GPU configuration

In [4]:
# Checking if GPU is available
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

CUDA available: True
GPU: Tesla T4
CUDA version: 12.6


#### Loading Model

In [5]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2  # binary sentiment
)

2026-02-07 12:03:34.718199: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770465814.951529      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770465815.020983      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770465815.587836      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770465815.587891      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770465815.587895      55 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Model tuning and optimization

#### Freezing encoder layers

In [6]:
for layer in model.bert.encoder.layer[-4:]:
    for param in layer.parameters():
        param.requires_grad = True

In [7]:
# Verifying that the encoder layers have been frozen
def count_trainable_params(model):
    total = 0
    trainable = 0
    for p in model.parameters():
        num = p.numel()
        total += num
        if p.requires_grad:
            trainable += num
    return total, trainable

total, trainable = count_trainable_params(model)
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"Trainable %:          {100 * trainable / total:.6f}%")

Total parameters:     109,483,778
Trainable parameters: 109,483,778
Trainable %:          100.000000%


#### Training upper layers using AdamW

In [8]:
# Importing libraries
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments

In [9]:
# Method for calculating metrics for later evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }


In [10]:
# Adding time logging functionality
import time
from transformers import TrainerCallback

class TimeCallback(TrainerCallback):
    def __init__(self):
        self.train_start = None
        self.epoch_start = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.train_start = time.time()

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.epoch_start = time.time()

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch_time = time.time() - self.epoch_start
        # state.epoch is a float sometimes; format nicely
        print(f"Epoch {state.epoch:.0f} time: {epoch_time:.2f} seconds")

    def on_train_end(self, args, state, control, **kwargs):
        total_time = time.time() - self.train_start
        print(f"Total training time: {total_time:.2f} seconds")


In [11]:
# Initializing trainer arguments
args = TrainingArguments(
    output_dir="/kaggle/working/baselineA_out",
    
    # evaluation + saving
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,

    # training
    num_train_epochs=3,
    learning_rate=3e-5,          # good for head-only training
    weight_decay=0.01,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    fp16=True,                   # T4 supports this well

    # logging
    logging_steps=50,
    report_to="none",
)


In [12]:
# Initializing trainer object with AdamW explicitly included
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=3e-5
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    optimizers=(optimizer, None),
    compute_metrics = compute_metrics,
    callbacks = [TimeCallback()]
)

In [13]:
print("Trainer device:", trainer.args.device)

Trainer device: cuda:0


In [14]:
# Initial training run
train_result = trainer.train()
val_result = trainer.evaluate()
print("Validation:", val_result)

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.165800,0.152860,0.943800,0.944013
2,0.108800,0.155030,0.947000,0.948110
3,0.051500,0.191371,0.945600,0.945752


Epoch 1 time: 1988.78 seconds


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch 2 time: 1989.99 seconds


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch 3 time: 1990.30 seconds
Total training time: 6226.70 seconds


/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Validation: {'eval_loss': 0.15503039956092834, 'eval_accuracy': 0.947, 'eval_f1': 0.9481104366555708, 'eval_runtime': 84.0375, 'eval_samples_per_second': 59.497, 'eval_steps_per_second': 0.94, 'epoch': 3.0}


In [15]:
test_result = trainer.evaluate(tokenized["test"])
print("Test:", test_result)

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Test: {'eval_loss': 0.16631144285202026, 'eval_accuracy': 0.9392, 'eval_f1': 0.941358024691358, 'eval_runtime': 84.3237, 'eval_samples_per_second': 59.295, 'eval_steps_per_second': 0.937, 'epoch': 3.0}


In [17]:
# Saving model
trainer.save_model("/kaggle/working/512_8to11_punchingit")
#tokenizer.save_pretrained("/kaggle/working/256_frozen_baseline")

## Simple Pipeline creation and testing

In [18]:
# Pipeline goes here
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.config.id2label = {0: "negative", 1: "positive"}
model.config.label2id = {"negative": 0, "positive": 1}

def predict_sentiment_with_confidence(
    text,
    model,
    tokenizer,
    device,
    max_length=256
):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)
        confidence, prediction = torch.max(probs, dim=1)

    return prediction.item(), confidence.item()


In [19]:
import os

os.listdir("/kaggle/input/256-tokenizer")

['256_tokenizer']

In [20]:
# Loading tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/512-tokenizer/512_tokenizer")

In [27]:
# Example
text = "The movie was not bad."

pred, conf = predict_sentiment_with_confidence(
    text, model, tokenizer, device
)

print(f"Predicted label: {pred}")
print(f"Confidence: {conf:.3f}")

Predicted label: 1
Confidence: 0.555
